# Stage 05 - RFdiffusion binder backbones against KRAS G12D pMHC

Generates de novo binder backbones against the **open** conformation of the
G12D decamer `VVVGADGVGK` on HLA-A\*11:01, where Asp6 projects out of the
groove. Route A of `docs/design_plan.md`.

**Target** `7ow3_L_G6D_chi1m70_chi2p120_relaxed_crop.pdb` - chain A = HLA
alpha1/alpha2 (residues 1-180), chain C = peptide (1-10). Built in stage 03 and
relaxed in stage 04. Its SHA-256 is checked in cell 3 against `TARGETS.tsv`.

**Setup cell reproduced from** `sokrypton/ColabDesign`,
`rf/examples/diffusion.ipynb` (RFdiffusion: Watson et al., *Nature* 620:1089,
2023). Its helper functions `run`, `fix_pdb`, `pdb_to_string` and `parse_pdb`
are reused here; `run_diffusion` is deliberately **not** used, so that every
hydra flag is visible in cell 4 rather than assembled out of sight.

## Manual steps (record these in `docs/colab_runbook.md`)

1. On the Mac: `python scripts/05a_stage_to_drive.py --drive-dir "<drive>/kras_pmhc_stage05"`
2. Wait for the Drive menu-bar icon to report sync complete.
3. Here: **Runtime -> Change runtime type -> T4 GPU**. Note which GPU was given.
4. Run cells 1-5. Cell 5 is the 8-design pilot.
5. On the Mac: `python scripts/05_ingest_backbones.py --drive-dir "<same>"` and
   read the pilot gate in `data/backbones/MANIFEST.tsv`.
6. **Only if the gate passes**, set `CUTOFF_UTC` in cell 1 and run cell 6.
7. Re-ingest on the Mac when the cutoff is reached.

The pilot gate is evaluated on the Mac, not here: it uses the Levy (2010) rSASA
classification from `kras_pmhc.interface`, and duplicating that in a notebook
would mean two implementations of the project's interface definition.

In [ ]:
#@title 1. Parameters - every knob lives here

# --- transport ---
DRIVE_FOLDER = "kras_pmhc_stage05"   # under My Drive; must match 05a_stage_to_drive.py
TARGET_NAME  = "7ow3_L_G6D_chi1m70_chi2p120_relaxed_crop.pdb"

# --- target geometry (see docs/design_plan.md) ---
TARGET_CONTIGS = ["A1-180/0", "C1-10/0"]  # HLA alpha1/alpha2, then peptide
BINDER_LENGTH  = "60-75"                   # sampled per design

# Hotspots centred on the mutation. C5/C6/C7 = Ala/Asp/Gly, so Asp6 is the only
# side chain of substance here; 3 hotspots is within the 3-6 the RFdiffusion
# authors recommend, and the model expects to make more contacts than specified.
HOTSPOTS = "C5,C6,C7"

# --- RFdiffusion sampling ---
# 0.5 per the RFdiffusion README for protein-protein interfaces: "we've
# consistently seen that reducing the noise added at inference improves
# designs... 0.5 is probably a good compromise". Costs topological diversity.
NOISE_SCALE = 0.5
DIFFUSER_T  = 50          # current upstream default
CHECKPOINT  = "Complex_base_ckpt.pt"   # set explicitly, not left to inference
USE_SEED    = True        # set False if cell 5 rejects inference.seed

# --- batching ---
BATCH_SIZE  = 8           # checkpoint granularity; a preemption loses at most this
PILOT_SEED  = 0

# --- production cutoff ---
# Colab runs in UTC. The competition deadline is Mon 9 Nov 06:59 EST
# (= 11:59 UTC). Generation must stop well before that because stages 06-11
# still have to run. Default below is 21:00 UTC Sun 8 Nov = 16:00 EST.
CUTOFF_UTC = "2026-11-08T21:00:00"

print(f"contigs : [{' '.join(TARGET_CONTIGS + [BINDER_LENGTH])}]")
print(f"hotspots: [{HOTSPOTS}]")
print(f"noise   : {NOISE_SCALE}  T={DIFFUSER_T}  ckpt={CHECKPOINT}")
print(f"cutoff  : {CUTOFF_UTC} UTC")

In [ ]:
#@title 2. Setup RFdiffusion (~3 min)
#
# Reproduced from sokrypton/ColabDesign rf/examples/diffusion.ipynb.
# Unmodified apart from this comment: it is well tested and the dependency
# pinning in it is load-bearing on Colab.
%%time
import os, time, signal
import sys, random, string, re
if not os.path.isdir("params"):
  os.system("apt-get install aria2")
  os.system("mkdir params")
  os.system("(\
  aria2c -q -x 16 https://files.ipd.uw.edu/krypton/schedules.zip; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/f572d396fae9206628714fb2ce00f72e/Complex_beta_ckpt.pt; \
  touch params/done.txt) &")

if not os.path.isdir("RFdiffusion"):
  print("installing RFdiffusion...")
  os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
  os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
  os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  os.system("cd RFdiffusion/env/SE3Transformer; pip install .")

if not os.path.isdir("colabdesign"):
  print("installing ColabDesign...")
  os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

if not os.path.isdir("RFdiffusion/models"):
  print("downloading RFdiffusion params...")
  os.system("mkdir RFdiffusion/models")
  models = ["Base_ckpt.pt","Complex_base_ckpt.pt","Complex_beta_ckpt.pt"]
  for m in models:
    while os.path.isfile(f"{m}.aria2"):
      time.sleep(5)
  os.system(f"mv {' '.join(models)} RFdiffusion/models")
  os.system("unzip schedules.zip; rm schedules.zip")

if 'RFdiffusion' not in sys.path:
  os.environ["DGLBACKEND"] = "pytorch"
  sys.path.append('RFdiffusion')

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets

from colabdesign.rf.utils import get_ca, fix_pdb
from colabdesign.shared.plot import plot_pseudo_3D

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 3. Mount Drive and load the target (checksum enforced)
import csv, hashlib, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive') / DRIVE_FOLDER
BACKBONE_DIR = DRIVE / 'backbones'
BACKBONE_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG = DRIVE / 'run_log.tsv'
TARGET_LOCAL = Path('/content/target.pdb')


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()


# TARGETS.tsv is written by scripts/05a_stage_to_drive.py and records the
# checksum of the stage-04 output. Verifying it here is what ties every
# generated backbone to a specific target: a half-synced or wrong file would
# otherwise produce plausible backbones against the wrong geometry.
with open(DRIVE / 'TARGETS.tsv') as handle:
    staged = {row['filename']: row for row in csv.DictReader(handle, delimiter='\t')}

if TARGET_NAME not in staged:
    raise SystemExit(f"{TARGET_NAME} not in TARGETS.tsv; staged: {sorted(staged)}")

record = staged[TARGET_NAME]
shutil.copy(DRIVE / TARGET_NAME, TARGET_LOCAL)
found = sha256(TARGET_LOCAL)
if found != record['sha256']:
    raise SystemExit(
        f"checksum mismatch for {TARGET_NAME}:\n"
        f"  expected {record['sha256']}\n  found    {found}\n"
        "Drive may still be syncing, or the wrong file was staged."
    )

print(f"target  : {TARGET_NAME}")
print(f"sha256  : {found[:16]}... verified against TARGETS.tsv")
print(f"staged  : {record['staged_utc']}")
print(f"outputs : {BACKBONE_DIR}")

In [ ]:
#@title 4. Generation driver
import subprocess
from datetime import datetime, timezone

CONTIGS = TARGET_CONTIGS + [BINDER_LENGTH]
CONTIG_STR = ' '.join(CONTIGS)
LOG_FIELDS = ('batch', 'seed', 'design', 'filename', 'contigs', 'hotspots',
              'noise_scale', 'diffuser_T', 'checkpoint', 'elapsed_s', 'written_utc')


def _log_rows():
    if not RUN_LOG.exists():
        return []
    with open(RUN_LOG) as handle:
        return list(csv.DictReader(handle, delimiter='\t'))


def _append_log(rows):
    exists = RUN_LOG.exists()
    with open(RUN_LOG, 'a', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=LOG_FIELDS, delimiter='\t')
        if not exists:
            writer.writeheader()
        writer.writerows(rows)


def next_batch():
    """Resume rather than restart: a preempted session picks up here."""
    rows = _log_rows()
    return (max(int(r['batch']) for r in rows) + 1) if rows else 0


def generate(n=BATCH_SIZE, seed=None, checkpoint=CHECKPOINT, hotspots=HOTSPOTS):
    """Run one RFdiffusion batch, writing backbones and a log row to Drive.

    Every flag is spelled out rather than assembled by run_diffusion, so the
    printed command is a complete record of what produced these coordinates.
    """
    batch = next_batch()
    seed = batch if seed is None else seed
    prefix = f'/content/outputs/batch{batch}'
    Path('/content/outputs').mkdir(exist_ok=True)

    flags = [
        f'inference.output_prefix={prefix}',
        f'inference.input_pdb={TARGET_LOCAL}',
        f'inference.num_designs={n}',
        f'inference.ckpt_override_path=./RFdiffusion/models/{checkpoint}',
        f'diffuser.T={DIFFUSER_T}',
        f'denoiser.noise_scale_ca={NOISE_SCALE}',
        f'denoiser.noise_scale_frame={NOISE_SCALE}',
        f"'contigmap.contigs=[{CONTIG_STR}]'",
        f"'ppi.hotspot_res=[{hotspots}]'",
    ]
    if USE_SEED:
        flags.append(f'inference.seed={seed}')

    cmd = './RFdiffusion/run_inference.py ' + ' '.join(flags)
    print(f'--- batch {batch} (seed {seed}, {n} designs) ---')
    print(cmd)

    started = time.time()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = time.time() - started

    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError(f'batch {batch} failed with code {result.returncode}')

    # fix_pdb re-splits RFdiffusion's single output chain into the chains the
    # contigs describe, so the target keeps A/C and the binder gets its own id.
    rows, written = [], []
    for i in range(n):
        source = Path(f'{prefix}_{i}.pdb')
        if not source.exists():
            print(f'  missing {source.name}; stopping this batch')
            break
        fixed = fix_pdb(source.read_text(), CONTIGS)
        name = f'batch{batch}_{i}.pdb'
        (BACKBONE_DIR / name).write_text(fixed)
        written.append(BACKBONE_DIR / name)
        rows.append({
            'batch': batch, 'seed': seed, 'design': i, 'filename': name,
            'contigs': CONTIG_STR, 'hotspots': hotspots,
            'noise_scale': NOISE_SCALE, 'diffuser_T': DIFFUSER_T,
            'checkpoint': checkpoint,
            'elapsed_s': round(elapsed / max(n, 1), 1),
            'written_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        })

    _append_log(rows)
    print(f'  wrote {len(written)} backbones in {elapsed/60:.1f} min '
          f'({elapsed/max(len(written),1):.0f} s each)')
    return written


print(f'contigs: [{CONTIG_STR}]')
print(f'resuming at batch {next_batch()}')

In [ ]:
#@title 5. Pilot - 8 designs, then STOP and ingest on the Mac
written = generate(n=BATCH_SIZE, seed=PILOT_SEED)

# Per-design Ca trace, coloured by the pLDDT-like confidence in the B-factor.
# This is an eyeball check only. The decision to continue is made on the Mac by
# scripts/05_ingest_backbones.py, which applies the Levy rSASA pilot gate.
for path in written:
    xyz, bfact = get_ca(str(path), get_bfact=True)
    fig = plt.figure(figsize=(4, 4), dpi=90)
    ax = fig.add_subplot(111); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(path.name, fontsize=9)
    plot_pseudo_3D(xyz, c=bfact, cmin=0.5, cmax=0.9, ax=ax)
    plt.show()

print('\nNow run on the Mac:')
print('  python scripts/05_ingest_backbones.py --drive-dir '
      f'"<drive>/{DRIVE_FOLDER}"')
print('Continue to cell 6 only if the pilot gate passes.')

In [ ]:
#@title 6. Production - batches of 8 until the cutoff
from datetime import datetime, timezone

cutoff = datetime.fromisoformat(CUTOFF_UTC).replace(tzinfo=timezone.utc)
print(f'generating until {cutoff.isoformat()} UTC')

while True:
    now = datetime.now(timezone.utc)
    remaining = (cutoff - now).total_seconds()
    if remaining <= 0:
        print('cutoff reached')
        break
    print(f'\n[{now.strftime("%H:%M:%S")}Z] {remaining/60:.0f} min left')
    try:
        generate(n=BATCH_SIZE)
    except RuntimeError as error:
        # One failed batch should not end the session: the log is already on
        # Drive and the next batch gets a fresh seed.
        print(f'batch failed, continuing: {error}')

rows = _log_rows()
print(f'\ntotal backbones on Drive: {len(rows)}')
print('Re-ingest on the Mac:')
print('  python scripts/05_ingest_backbones.py --drive-dir '
      f'"<drive>/{DRIVE_FOLDER}"')